# bridgechem — a hard-sphere gas in a box

This notebook shows the milestone-1 core: an ideal-ish gas of hard spheres that
fly ballistically and collide elastically with each other and the walls.

Everything is computed in **SI units**. Lengths are entered in **nm** and masses
in **amu** for convenience; results (speeds, temperature, pressure) come back in SI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bridgechem as bc

system = bc.box(N=800, size=(80, 80), temperature=300)  # 80 nm x 80 nm of argon
system

## Run and watch it

`run` integrates the system (numba-accelerated) and returns a `Simulation` holding
the trajectory. `show()` replays it as an animation.

In [ ]:
sim = system.run(steps=20000)
sim.show(color_by="speed")  # in Jupyter: from IPython.display import HTML; HTML(sim.show().to_jshtml())

## Speed distribution vs Maxwell–Boltzmann

The histogram of speeds should match the 2D Maxwell–Boltzmann (Rayleigh)
distribution — and the agreement improves as `N` grows.

In [ ]:
sim.histogram("speeds", frame="all")
plt.show()

## Pressure and the (2D) ideal-gas law

Pressure is measured from the momentum the particles hand to the walls. Compare it
with $P = N k_B T / A$. The small positive excess is real physics: finite-size
particles have an excluded area, just like a van der Waals gas.

In [ ]:
P = sim.calculate("pressure")
P_ideal = sim.ideal_gas_pressure()
print(f"measured P = {P:.3e} N/m")
print(f"ideal    P = {P_ideal:.3e} N/m")
print(f"ratio      = {P / P_ideal:.3f}")

## Watch a distribution *relax* to Maxwell–Boltzmann

Start every particle at the same speed (random directions). Collisions alone drive
the system to the Maxwell–Boltzmann distribution — a nice illustration of how
equilibrium emerges.

In [ ]:
system2 = bc.box(N=800, size=(80, 80), temperature=300, velocity_init="uniform_speed")
sim2 = system2.run(steps=20000)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
sim2.histogram("speeds", frame=0, ax=axes[0]); axes[0].set_title("initial (uniform speed)")
sim2.histogram("speeds", frame=-1, ax=axes[1]); axes[1].set_title("after collisions")
plt.show()

## Coming next

- `system.add_interactions("LJ")` — Lennard-Jones / dispersion forces with
  velocity-Verlet, virial pressure and phase transitions.
- `system.set_temperature(...)` — gradual cooling to explore condensation.
- 3D boxes.